In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pickle
import os

# Check PyTorch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load master dataset
df = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['community', 'date']).reset_index(drop=True)

print(f"\nDataset shape: {df.shape}")
print(f"Communities: {df['community'].nunique()}")
print(f"Drought cases: {df['drought_label'].sum()}")

PyTorch version: 2.12.1+cpu
CUDA available: False
Using device: cpu

Dataset shape: (1575, 17)
Communities: 15
Drought cases: 29


In [3]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

# Encode community and region
le_community = LabelEncoder()
le_region = LabelEncoder()
df['community_enc'] = le_community.fit_transform(df['community'])
df['region_enc'] = le_region.fit_transform(df['region'])

features = ['community_enc', 'region_enc', 'ndvi', 'rainfall_mm',
            'temp_max', 'temp_min', 'humidity', 'et0',
            'lst_celsius', 'ndvi_anomaly', 'rainfall_deficit',
            'water_balance', 'spei_proxy']

# Scale features to 0-1 range (LSTM requires normalised input)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[features])
y = df['drought_label'].values

# Create sequences — LSTM looks back 4 weeks to predict current week
SEQUENCE_LENGTH = 4

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    # Create sequences per community to avoid mixing communities
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

# Build sequences per community
all_X, all_y = [], []
for community in df['community'].unique():
    mask = df['community'] == community
    X_comm = X_scaled[mask]
    y_comm = y[mask]
    if len(X_comm) > SEQUENCE_LENGTH:
        X_seq, y_seq = create_sequences(X_comm, y_comm, SEQUENCE_LENGTH)
        all_X.append(X_seq)
        all_y.append(y_seq)

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)

print(f"Sequence dataset shape: {X_all.shape}")
print(f"Labels shape: {y_all.shape}")
print(f"Drought cases: {y_all.sum()}")
print(f"Sequence length: {SEQUENCE_LENGTH} weeks lookback")

Sequence dataset shape: (1515, 4, 13)
Labels shape: (1515,)
Drought cases: 28
Sequence length: 4 weeks lookback


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# Train/test split — temporal: train on first 80%, test on last 20%
split_idx = int(len(X_all) * 0.8)
X_train = X_all[:split_idx]
X_test  = X_all[split_idx:]
y_train = y_all[:split_idx]
y_test  = y_all[split_idx:]

print(f"Training sequences: {len(X_train)} | Drought: {y_train.sum()}")
print(f"Test sequences:     {len(X_test)}  | Drought: {y_test.sum()}")

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).to(device)
X_test_t  = torch.FloatTensor(X_test).to(device)
y_train_t = torch.FloatTensor(y_train).to(device)
y_test_t  = torch.FloatTensor(y_test).to(device)

# Class weights for imbalance
drought_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nDrought class weight: {drought_weight:.1f}x")

# ── LSTM MODEL ──────────────────────────────────────────
class DroughtLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(DroughtLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Take last timestep output
        out = lstm_out[:, -1, :]
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return self.sigmoid(out).squeeze()

# Model config
INPUT_SIZE  = X_all.shape[2]  # 13 features
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
DROPOUT     = 0.3
EPOCHS      = 100
BATCH_SIZE  = 32
LR          = 0.001

model = DroughtLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
print(f"\nLSTM Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Loss and optimizer with class weighting
pos_weight = torch.tensor([drought_weight]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5, verbose=True)

# DataLoader
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ── TRAINING LOOP ───────────────────────────────────────
print("\nTraining LSTM...")
train_losses = []
best_auc = 0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Evaluate every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            y_prob = model(X_test_t).cpu().numpy()
            if len(np.unique(y_test)) > 1:
                auc = roc_auc_score(y_test, y_prob)
                if auc > best_auc:
                    best_auc = auc
                    best_model_state = model.state_dict().copy()
                print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | AUC: {auc:.3f}")
            else:
                print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")

    scheduler.step(avg_loss)

print(f"\nBest AUC-ROC: {best_auc:.3f}")
print("LSTM training complete!")

Training sequences: 1212 | Drought: 24
Test sequences:     303  | Drought: 4

Drought class weight: 49.5x

LSTM Model Architecture:
DroughtLSTM(
  (lstm): LSTM(13, 64, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

Total parameters: 55,617


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [5]:
# Loss with class weighting
pos_weight = torch.tensor([drought_weight]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5)  # removed verbose=True

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Training LSTM...")
train_losses = []
best_auc = 0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            y_prob = model(X_test_t).cpu().numpy()
            if len(np.unique(y_test)) > 1:
                auc = roc_auc_score(y_test, y_prob)
                if auc > best_auc:
                    best_auc = auc
                    best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
                print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | AUC: {auc:.3f}")
            else:
                print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")

print(f"\nBest AUC-ROC: {best_auc:.3f}")
print("LSTM training complete!")

Training LSTM...
Epoch  10 | Loss: 1.2684 | AUC: 0.907
Epoch  20 | Loss: 1.2297 | AUC: 0.962
Epoch  30 | Loss: 1.1719 | AUC: 0.966
Epoch  40 | Loss: 1.1395 | AUC: 0.962
Epoch  50 | Loss: 1.1520 | AUC: 0.965
Epoch  60 | Loss: 1.1503 | AUC: 0.966
Epoch  70 | Loss: 1.1225 | AUC: 0.961
Epoch  80 | Loss: 1.1211 | AUC: 0.960
Epoch  90 | Loss: 1.1147 | AUC: 0.960
Epoch 100 | Loss: 1.1149 | AUC: 0.961

Best AUC-ROC: 0.966
LSTM training complete!


In [6]:
# Load best model
model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad():
    y_prob_lstm = model(X_test_t).cpu().numpy()
    y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

print("=== LSTM Evaluation ===")
print(classification_report(y_test, y_pred_lstm,
      target_names=['No Drought', 'Drought']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_lstm):.3f}")

# Save model
import pickle
torch.save(model.state_dict(),
    'C:/Users/ELITE/Documents/AGROALERT/src_model/lstm_model.pth')
with open('C:/Users/ELITE/Documents/AGROALERT/src_model/scaler.pkl','wb') as f:
    pickle.dump(scaler, f)
print("LSTM model saved!")

=== LSTM Evaluation ===
              precision    recall  f1-score   support

  No Drought       1.00      0.94      0.97       299
     Drought       0.14      0.75      0.24         4

    accuracy                           0.94       303
   macro avg       0.57      0.84      0.60       303
weighted avg       0.99      0.94      0.96       303

AUC-ROC: 0.966
LSTM model saved!


In [7]:
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load RF predictions
df_pred = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/predictions.csv')

# Load master dataset
df = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

# RF scores already in df_pred
# Get LSTM scores for same records
le_c = LabelEncoder()
le_r = LabelEncoder()
df['community_enc'] = le_c.fit_transform(df['community'])
df['region_enc'] = le_r.fit_transform(df['region'])

features = ['community_enc','region_enc','ndvi','rainfall_mm',
            'temp_max','temp_min','humidity','et0',
            'lst_celsius','ndvi_anomaly','rainfall_deficit',
            'water_balance','spei_proxy']

X_scaled_all = scaler.transform(df[features])

# Build sequences for full dataset
all_X_full, all_idx = [], []
for community in df['community'].unique():
    mask = np.where(df['community'].values == community)[0]
    X_comm = X_scaled_all[mask]
    if len(X_comm) > SEQUENCE_LENGTH:
        for i in range(SEQUENCE_LENGTH, len(X_comm)):
            all_X_full.append(X_comm[i-SEQUENCE_LENGTH:i])
            all_idx.append(mask[i])

X_full_t = torch.FloatTensor(np.array(all_X_full)).to(device)

model.eval()
with torch.no_grad():
    lstm_probs = model(X_full_t).cpu().numpy()

# Map LSTM scores back to master dataset
df['lstm_score'] = np.nan
for i, idx in enumerate(all_idx):
    df.iloc[idx, df.columns.get_loc('lstm_score')] = lstm_probs[i]

df['lstm_score'] = df['lstm_score'].fillna(0)

# Merge with predictions
df_pred['lstm_score'] = df['lstm_score'].values

# Three-component ensemble
df_pred['ensemble_score_v2'] = (
    0.4 * df_pred['rf_score'] +
    0.4 * df_pred['lstm_score'] +
    0.2 * df_pred['rule_score']
)
df_pred['alert_v2'] = (df_pred['ensemble_score_v2'] >= 0.65).astype(int)

print("=== Three-Component Ensemble Results ===")
print(f"RF + LSTM + Rule alerts: {df_pred['alert_v2'].sum()}")
print(f"Previous RF + Rule alerts: {df_pred['alert_triggered'].sum()}")
print("\nRisk by community (new ensemble):")
summary = df_pred.groupby('community')['ensemble_score_v2'].max().sort_values(ascending=False)
for comm, score in summary.items():
    status = "HIGH" if score >= 0.75 else "MEDIUM" if score >= 0.5 else "LOW"
    print(f"  {comm}: {score:.3f} — {status}")

df_pred.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/predictions.csv', index=False)
print("\nUpdated predictions saved!")

=== Three-Component Ensemble Results ===
RF + LSTM + Rule alerts: 21
Previous RF + Rule alerts: 26

Risk by community (new ensemble):
  Sunyani: 0.939 — HIGH
  Goaso: 0.939 — HIGH
  Koforidua: 0.939 — HIGH
  Damongo: 0.938 — HIGH
  Ho: 0.937 — HIGH
  Kumasi: 0.937 — HIGH
  Techiman: 0.911 — HIGH
  Nalerigu: 0.625 — MEDIUM
  Tamale: 0.623 — MEDIUM
  Bolgatanga: 0.611 — MEDIUM
  Dambai: 0.544 — MEDIUM
  Sekondi-Takoradi: 0.537 — MEDIUM
  Sefwi Wiawso: 0.485 — LOW
  Wa: 0.406 — LOW
  Cape Coast: 0.092 — LOW

Updated predictions saved!


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

df = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

le_c = LabelEncoder()
le_r = LabelEncoder()
df['community_enc'] = le_c.fit_transform(df['community'])
df['region_enc'] = le_r.fit_transform(df['region'])

features = ['community_enc','region_enc','ndvi','rainfall_mm',
            'temp_max','temp_min','humidity','et0',
            'lst_celsius','ndvi_anomaly','rainfall_deficit',
            'water_balance','spei_proxy']

# Temporal split: train on 2022, test on 2023
train_mask = df['date'].dt.year == 2022
test_mask  = df['date'].dt.year == 2023

X_train = df[train_mask][features]
y_train = df[train_mask]['drought_label']
X_test  = df[test_mask][features]
y_test  = df[test_mask]['drought_label']

print(f"Temporal split:")
print(f"  Train (2022): {len(X_train)} records | {y_train.sum()} drought")
print(f"  Test  (2023): {len(X_test)} records  | {y_test.sum()} drought")

rf_temporal = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_split=5, class_weight='balanced',
    random_state=42)
rf_temporal.fit(X_train, y_train)

y_prob = rf_temporal.predict_proba(X_test)[:, 1]
y_pred = rf_temporal.predict(X_test)

print("\n=== Temporal Validation Results ===")
print(classification_report(y_test, y_pred,
      target_names=['No Drought','Drought']))
if y_test.sum() > 0:
    auc = roc_auc_score(y_test, y_prob)
    print(f"Temporal AUC-ROC: {auc:.3f}")

Temporal split:
  Train (2022): 780 records | 10 drought
  Test  (2023): 795 records  | 19 drought

=== Temporal Validation Results ===
              precision    recall  f1-score   support

  No Drought       0.99      1.00      0.99       776
     Drought       1.00      0.58      0.73        19

    accuracy                           0.99       795
   macro avg       0.99      0.79      0.86       795
weighted avg       0.99      0.99      0.99       795

Temporal AUC-ROC: 0.994


In [14]:
# Temporal split
train_mask = df['date'].dt.year == 2022
test_mask  = df['date'].dt.year == 2023

# Build sequences per year
def build_sequences_for_mask(df, X_scaled, mask_bool, seq_len=4):
    all_X, all_y = [], []
    for community in df['community'].unique():
        comm_mask = (df['community'] == community) & mask_bool
        idx = np.where(comm_mask)[0]
        X_c = X_scaled[idx]
        y_c = df['drought_label'].values[idx]
        if len(X_c) > seq_len:
            for i in range(seq_len, len(X_c)):
                all_X.append(X_c[i-seq_len:i])
                all_y.append(y_c[i])
    return np.array(all_X), np.array(all_y)

X_train_t_seq, y_train_t_seq = build_sequences_for_mask(
    df, X_scaled_all, df['date'].dt.year == 2022)
X_test_t_seq, y_test_t_seq = build_sequences_for_mask(
    df, X_scaled_all, df['date'].dt.year == 2023)

print(f"Train 2022: {len(X_train_t_seq)} sequences | Drought: {y_train_t_seq.sum()}")
print(f"Test  2023: {len(X_test_t_seq)} sequences  | Drought: {y_test_t_seq.sum()}")

# Train new temporal model
model_temporal = DroughtLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
optimizer_t = torch.optim.Adam(model_temporal.parameters(), lr=LR)

Xt = torch.FloatTensor(X_train_t_seq).to(device)
yt = torch.FloatTensor(y_train_t_seq).to(device)
ds = TensorDataset(Xt, yt)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

dw = torch.tensor([(y_train_t_seq==0).sum()/(y_train_t_seq==1).sum()+1e-8]).to(device)
crit = nn.BCEWithLogitsLoss(pos_weight=dw)

for epoch in range(EPOCHS):
    model_temporal.train()
    for Xb, yb in dl:
        optimizer_t.zero_grad()
        loss = crit(model_temporal(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_temporal.parameters(), 1.0)
        optimizer_t.step()

# Evaluate
model_temporal.eval()
with torch.no_grad():
    Xtest_t = torch.FloatTensor(X_test_t_seq).to(device)
    y_prob_t = model_temporal(Xtest_t).cpu().numpy()

print("\n=== Temporal Validation (Train 2022 → Test 2023) ===")
if y_test_t_seq.sum() > 0:
    print(f"ROC-AUC: {roc_auc_score(y_test_t_seq, y_prob_t):.3f}")
    print(f"PR-AUC:  {average_precision_score(y_test_t_seq, y_prob_t):.3f}")
else:
    print("No drought cases in 2023 test set — check label distribution")


Train 2022: 720 sequences | Drought: 9
Test  2023: 735 sequences  | Drought: 12

=== Temporal Validation (Train 2022 → Test 2023) ===
ROC-AUC: 0.938
PR-AUC:  0.163


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['community','date']).reset_index(drop=True)

le_c = LabelEncoder()
le_r = LabelEncoder()
df['community_enc'] = le_c.fit_transform(df['community'])
df['region_enc']    = le_r.fit_transform(df['region'])

# RAW FEATURES ONLY — NDVI anomaly and SPEI proxy excluded
# to reduce target construction leakage
features_raw = [
    'community_enc', 'region_enc',
    'ndvi', 'rainfall_mm', 'temp_max', 'temp_min',
    'humidity', 'et0', 'lst_celsius',
    'water_balance', 'rainfall_deficit'
]
# Note: NDVI anomaly and SPEI proxy removed because they directly
# define the drought label — including them causes target construction leakage

print(f"Features used: {len(features_raw)}")
print("Excluded: ndvi_anomaly, spei_proxy (used in label construction)")

X_raw = df[features_raw].values
y     = df['drought_label'].values

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

# Temporal split — train 2022, test 2023
train_mask = df['date'].dt.year == 2022
test_mask  = df['date'].dt.year == 2023

X_train_raw = X_scaled[train_mask]
X_test_raw  = X_scaled[test_mask]
y_train     = y[train_mask]
y_test      = y[test_mask]

print(f"\nTemporal split:")
print(f"  Train (2022): {len(X_train_raw)} | Drought: {y_train.sum()}")
print(f"  Test  (2023): {len(X_test_raw)}  | Drought: {y_test.sum()}")

Features used: 11
Excluded: ndvi_anomaly, spei_proxy (used in label construction)

Temporal split:
  Train (2022): 780 | Drought: 10
  Test  (2023): 795  | Drought: 19


In [2]:
# ── BASELINE 1: Logistic Regression ──────────────────────
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_raw, y_train)
lr_prob = lr.predict_proba(X_test_raw)[:, 1]

lr_roc = roc_auc_score(y_test, lr_prob) if y_test.sum() > 0 else 0
lr_pr  = average_precision_score(y_test, lr_prob) if y_test.sum() > 0 else 0
print(f"Logistic Regression — ROC-AUC: {lr_roc:.3f} | PR-AUC: {lr_pr:.3f}")

# ── BASELINE 2: Random Forest ─────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=42)
rf.fit(X_train_raw, y_train)
rf_prob = rf.predict_proba(X_test_raw)[:, 1]

rf_roc = roc_auc_score(y_test, rf_prob) if y_test.sum() > 0 else 0
rf_pr  = average_precision_score(y_test, rf_prob) if y_test.sum() > 0 else 0
print(f"Random Forest       — ROC-AUC: {rf_roc:.3f} | PR-AUC: {rf_pr:.3f}")

print("\nBaseline models done.")

Logistic Regression — ROC-AUC: 0.806 | PR-AUC: 0.094
Random Forest       — ROC-AUC: 0.828 | PR-AUC: 0.212

Baseline models done.


In [3]:
device = torch.device('cpu')
SEQUENCE_LENGTH = 4
INPUT_SIZE  = len(features_raw)
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
DROPOUT     = 0.3
EPOCHS      = 100
BATCH_SIZE  = 32
LR          = 0.001

# Build sequences per community — temporal split
def build_sequences(df, X_scaled, year, seq_len=4):
    all_X, all_y = [], []
    for comm in df['community'].unique():
        mask = (df['community'] == comm) & (df['date'].dt.year == year)
        idx  = np.where(mask)[0]
        Xc   = X_scaled[idx]
        yc   = df['drought_label'].values[idx]
        for i in range(seq_len, len(Xc)):
            all_X.append(Xc[i-seq_len:i])
            all_y.append(yc[i])
    return np.array(all_X), np.array(all_y)

X_tr, y_tr = build_sequences(df, X_scaled, 2022)
X_te, y_te = build_sequences(df, X_scaled, 2023)

print(f"LSTM sequences — Train: {len(X_tr)} | Drought: {y_tr.sum()}")
print(f"LSTM sequences — Test:  {len(X_te)} | Drought: {y_te.sum()}")

class DroughtLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.fc1  = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(32, 1)
        self.sig  = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.drop(out[:, -1, :])
        return self.sig(self.fc2(self.relu(self.fc1(out)))).squeeze()

model = DroughtLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)

dw   = torch.tensor([(y_tr==0).sum()/(y_tr==1).sum()+1e-8]).to(device)
crit = nn.BCEWithLogitsLoss(pos_weight=dw)
opt  = torch.optim.Adam(model.parameters(), lr=LR)
sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)

Xt = torch.FloatTensor(X_tr).to(device)
yt = torch.FloatTensor(y_tr).to(device)
ds = TensorDataset(Xt, yt)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

best_pr  = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    ep_loss = 0
    for Xb, yb in dl:
        opt.zero_grad()
        loss = crit(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        ep_loss += loss.item()
    sch.step(ep_loss/len(dl))

    if (epoch+1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            yp = model(torch.FloatTensor(X_te).to(device)).cpu().numpy()
        if y_te.sum() > 0:
            pr = average_precision_score(y_te, yp)
            rc = roc_auc_score(y_te, yp)
            if pr > best_pr:
                best_pr = pr
                best_state = {k: v.clone() for k,v in model.state_dict().items()}
            print(f"Epoch {epoch+1:3d} | Loss: {ep_loss/len(dl):.4f} | ROC-AUC: {rc:.3f} | PR-AUC: {pr:.3f}")

# Final evaluation
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    lstm_prob = model(torch.FloatTensor(X_te).to(device)).cpu().numpy()

lstm_roc = roc_auc_score(y_te, lstm_prob) if y_te.sum() > 0 else 0
lstm_pr  = average_precision_score(y_te, lstm_prob) if y_te.sum() > 0 else 0
lstm_pred = (lstm_prob >= 0.5).astype(int)

print(f"\n=== FINAL RESULTS (Temporal: Train 2022 → Test 2023) ===")
print(f"\nRandom baseline PR-AUC (prevalence): {y_te.sum()/len(y_te):.3f}")
print(f"\nLogistic Regression — ROC: {lr_roc:.3f} | PR: {lr_pr:.3f}")
print(f"Random Forest       — ROC: {rf_roc:.3f} | PR: {rf_pr:.3f}")
print(f"LSTM (DroughtLSTM)  — ROC: {lstm_roc:.3f} | PR: {lstm_pr:.3f}")
print(f"\nLSTM PR-AUC vs random baseline: {lstm_pr/max(y_te.sum()/len(y_te),1e-8):.1f}x")
print(f"\nLSTM Classification Report:")
print(classification_report(y_te, lstm_pred, target_names=['No Drought','Drought']))

# Save
torch.save(model.state_dict(),
    'C:/Users/ELITE/Documents/AGROALERT/src_model/lstm_model_v2.pth')
print("Model saved!")

LSTM sequences — Train: 720 | Drought: 9
LSTM sequences — Test:  735 | Drought: 12
Epoch  20 | Loss: 1.1708 | ROC-AUC: 0.877 | PR-AUC: 0.084
Epoch  40 | Loss: 1.1977 | ROC-AUC: 0.905 | PR-AUC: 0.098
Epoch  60 | Loss: 1.1580 | ROC-AUC: 0.917 | PR-AUC: 0.107
Epoch  80 | Loss: 1.1103 | ROC-AUC: 0.908 | PR-AUC: 0.105
Epoch 100 | Loss: 1.0619 | ROC-AUC: 0.898 | PR-AUC: 0.134

=== FINAL RESULTS (Temporal: Train 2022 → Test 2023) ===

Random baseline PR-AUC (prevalence): 0.016

Logistic Regression — ROC: 0.806 | PR: 0.094
Random Forest       — ROC: 0.828 | PR: 0.212
LSTM (DroughtLSTM)  — ROC: 0.898 | PR: 0.134

LSTM PR-AUC vs random baseline: 8.2x

LSTM Classification Report:
              precision    recall  f1-score   support

  No Drought       0.98      0.99      0.99       723
     Drought       0.09      0.08      0.09        12

    accuracy                           0.97       735
   macro avg       0.54      0.53      0.54       735
weighted avg       0.97      0.97      0.97       